# 07 Project Risk Scoring

Classification models to predict high-realization vs low-realization projects, plus transparent scoring of maturity and delay risk.

> Run the pipeline first: `python run_pipeline.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

import json
pipeline = pd.read_csv(processed / 'reactor_pipeline.csv')
risk = pd.read_csv(predictions / 'project_risk_scores.csv')
metrics = json.loads((ROOT / 'outputs' / 'metrics' / 'risk_model_metrics.json').read_text())
print("Best model:", metrics['best_model'])
pd.DataFrame(metrics['models']).T.drop(columns=['notes'])

## Label distribution

The binary label `label_high_realization` is derived from `realization_probability >= 0.6`. Since `realization_probability` comes from `project_maturity_score`, **this is a circular exercise** — the models learn to reproduce the scoring rule. The value is in demonstrating the classification pipeline (preprocessing, model comparison, metrics), not the predictive validity.

In [ ]:
pipeline['label'] = (pipeline['realization_probability'] >= 0.6).astype(int)
label_counts = pipeline.groupby(['status_group','label']).size().unstack(fill_value=0)
label_counts.columns = ['Low realization','High realization']
print(label_counts)

fig, ax = plt.subplots(figsize=(8,4))
label_counts.plot.bar(ax=ax, colormap='RdYlGn')
ax.set_title('Realization Label by Status Group')
ax.set_xlabel(''); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## Risk matrix: maturity vs delay risk

In [ ]:
fig = px.scatter(
    risk,
    x='project_maturity_score', y='delay_risk_score',
    size='capacity_mwe', color='risk_level',
    hover_name='reactor_name',
    hover_data=['country','technology_family','realization_label'],
    color_discrete_map={'Low':'green','Medium':'orange','High':'red'},
    title='Project Risk Matrix — Maturity vs Delay Risk',
    template='plotly_white'
)
fig.show()

## Score distributions by technology

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tech_maturity = risk.groupby('technology_family')['project_maturity_score'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=tech_maturity, x='technology_family', y='project_maturity_score', ax=axes[0], palette='Blues_d')
axes[0].set_title('Avg Project Maturity Score by Technology')
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=30)

tech_risk = risk.groupby('technology_family')['delay_risk_score'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=tech_risk, x='technology_family', y='delay_risk_score', ax=axes[1], palette='Reds_d')
axes[1].set_title('Avg Delay Risk Score by Technology')
axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

## What the scores represent

| Score | Components |
|---|---|
| `project_maturity_score` | 45% status stage + 25% technology maturity + 20% country experience + 10% GDP proxy |
| `delay_risk_score` | 100 - maturity + penalty for large units + penalty for proposed status |
| `realization_probability` | maturity score / 100, clipped to [0.05, 0.95] |

These are **transparent decision-support heuristics**, not statistical predictions. The lack of historical realization labels makes supervised validation impossible with current data.

## Sensitivity: what drives score changes

In [ ]:
import numpy as np

# Show how maturity score components weight different status groups
status_weights = {'Construction': 85, 'Planned': 55, 'Proposed': 30, 'Paused': 20}
experience_examples = [0, 20, 50, 70]
tech_score = 55  # mid-range technology

print(f"{'Status':<15} {'Exp=0':>10} {'Exp=20':>10} {'Exp=50':>10} {'Exp=70':>10}")
print('-' * 55)
for status, stage in status_weights.items():
    scores = []
    for exp in experience_examples:
        exp_norm = np.clip(exp / 70 * 100, 0, 100)
        score = 0.45 * stage + 0.25 * tech_score + 0.20 * exp_norm + 0.10 * 60  # mid GDP
        scores.append(round(score, 1))
    print(f"{status:<15} {scores[0]:>10} {scores[1]:>10} {scores[2]:>10} {scores[3]:>10}")